In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from sqlalchemy import create_engine
import os

def connect_to_db():
    engine = create_engine(
        f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@"
        f"{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
    )
    return engine

engine = connect_to_db()

conn = engine.connect()

In [ ]:
df = pd.read_sql("SELECT * FROM public.gold_model_dataset", engine)

df.head()

In [ ]:
df.info()
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df[df["stay_duration"] <= 0].shape
df[df["advance_booking_days"] < 0].shape
df[df["total_guests"] <= 0].shape

In [ ]:
num_cols = [
    "stay_duration",
    "advance_booking_days",
    "orig_destination_distance",
    "cnt",
    "total_guests"
]

plt.figure(figsize=(12,6))
sns.boxplot(data=df[num_cols])
plt.title("Boxplot - Outliers Detection")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df[num_cols].hist(figsize=(12,8))
plt.tight_layout()

In [ ]:
for col in num_cols:
    p1, p99 = np.percentile(df[col], [1, 99])
    print(f"{col} → P1: {p1:.2f}, P99: {p99:.2f}")

In [ ]:
df_outliers = df[
    (df["stay_duration"] > 30) |
    (df["advance_booking_days"] > 365) |
    (df["orig_destination_distance"] > 10000)
]

df_outliers.head()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df["srch_destination_id"].isnull()].shape
df[df["hotel_cluster"].isnull()].shape

In [ ]:
df["long_stay"] = df["stay_duration"] > df["stay_duration"].quantile(0.95)

df.groupby("long_stay")["is_booking"].mean()

In [ ]:
(df["orig_destination_distance"] == -1).sum()

In [ ]:
conn.close()

## Key Insights

- The dataset presents overall good quality, with minimal missing values after preprocessing.
- Several numerical variables (e.g., `stay_duration`, `advance_booking_days`, and `orig_destination_distance`) exhibit skewed distributions and the presence of outliers.
- Extreme values in certain variables appear to reflect real user behavior rather than data errors (e.g., long booking windows or extended stays).
- Placeholder values (such as -1 in `orig_destination_distance`) represent unknown information and were treated explicitly rather than removed.
- No significant duplication or structural inconsistencies were identified in the dataset.

## Data Quality Decisions

- Outliers were not removed automatically, as they may represent valid behavioral patterns.
- Skewed variables were retained for further analysis, with potential transformation considered in later stages if needed.
- Missing or placeholder values were handled as separate categories to preserve information.
- The dataset was deemed reliable and suitable for downstream analysis and modeling.

## Conclusion

The dataset is clean and consistent, with no critical data quality issues. While some variables present skewness and extreme values, these are likely inherent to user behavior and should be handled thoughtfully rather than discarded. Overall, the data is well-prepared for further analytical steps.